# TP Méthodes d'Ensemble - Partie 3 : Boosting & XGBoost

Découvrir la puissance du Boosting et manipuler la librairie XGBoost.

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb

## 0. Chargement des données

Prérequis : voir `README.md`.

In [9]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
           'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

df = pd.read_csv(url, names=columns, na_values='?')
df = df.dropna()
df['target'] = (df['target'] > 0).astype(int)

X = df.drop('target', axis=1).values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset chargé : {X_train.shape[0]} train, {X_test.shape[0]} test")

Dataset chargé : 237 train, 60 test


## 1. Gradient Boosting avec Sklearn

**GradientBoostingClassifier** : entraînement séquentiel, chaque arbre corrige les erreurs des précédents.

### TODO 1 : Modèle de base (paramètres par défaut)

In [10]:
gb_default = GradientBoostingClassifier(random_state=42)
gb_default.fit(X_train, y_train)
acc_default = accuracy_score(y_test, gb_default.predict(X_test))
print(f"GradientBoostingClassifier (défaut) : accuracy = {acc_default:.4f}")

GradientBoostingClassifier (défaut) : accuracy = 0.7667


### TODO 2 : Impact du learning_rate (0.01, 0.1, 1.0)

Plus le learning rate est **faible**, plus il faut d'arbres pour converger, mais on limite le surapprentissage.  
Un learning rate **élevé** (ex. 1.0) peut converger vite mais risque de surajuster.

In [11]:
for lr in [0.01, 0.1, 1.0]:
    gb = GradientBoostingClassifier(learning_rate=lr, n_estimators=100, random_state=42)
    gb.fit(X_train, y_train)
    acc = accuracy_score(y_test, gb.predict(X_test))
    print(f"  learning_rate={lr:.2f}  ->  accuracy = {acc:.4f}")

  learning_rate=0.01  ->  accuracy = 0.8167
  learning_rate=0.10  ->  accuracy = 0.7667
  learning_rate=1.00  ->  accuracy = 0.8167


### TODO 3 : Impact de n_estimators (10, 100, 500)

Plus d'arbres améliore en général la performance, mais avec un fort learning rate le risque de **surapprentissage** augmente.

In [12]:
for n in [10, 100, 500]:
    gb = GradientBoostingClassifier(n_estimators=n, learning_rate=0.1, random_state=42)
    gb.fit(X_train, y_train)
    acc = accuracy_score(y_test, gb.predict(X_test))
    print(f"  n_estimators={n:3d}  ->  accuracy = {acc:.4f}")

  n_estimators= 10  ->  accuracy = 0.8333
  n_estimators=100  ->  accuracy = 0.7667
  n_estimators=500  ->  accuracy = 0.8000


## 2. XGBoost (Extreme Gradient Boosting)

Librairie optimisée : rapidité, régularisation L1/L2, early stopping, gestion des valeurs manquantes.

In [13]:
model_xgb = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    use_label_encoder=False,
    eval_metric='logloss'
)

model_xgb.fit(X_train_scaled, y_train)
y_pred_xgb = model_xgb.predict(X_test_scaled)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"XGBClassifier : accuracy = {acc_xgb:.4f}")

XGBClassifier : accuracy = 0.8167


/Users/gonthierlucas/miniconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [09:50:14] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


## 3. Early Stopping

On fournit un **eval_set** (ici le test set pour l'exemple ; en pratique, utiliser un set de validation dédié).  
L'entraînement s'arrête si la métrique sur eval_set ne s'améliore plus pendant `early_stopping_rounds` rounds.

In [14]:
eval_set = [(X_test_scaled, y_test)]

model_xgb_es = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    early_stopping_rounds=10,
    eval_metric='logloss'
)

model_xgb_es.fit(
    X_train_scaled,
    y_train,
    eval_set=eval_set,
    verbose=True
)

best_iteration = model_xgb_es.best_iteration
print(f"\nEntraînement arrêté à l'itération (round) : {best_iteration}")

[0]	validation_0-logloss:0.64860
[1]	validation_0-logloss:0.61338
[2]	validation_0-logloss:0.58637
[3]	validation_0-logloss:0.56114
[4]	validation_0-logloss:0.53477
[5]	validation_0-logloss:0.51733
[6]	validation_0-logloss:0.49668
[7]	validation_0-logloss:0.48445
[8]	validation_0-logloss:0.47684
[9]	validation_0-logloss:0.46275
[10]	validation_0-logloss:0.45518
[11]	validation_0-logloss:0.44636
[12]	validation_0-logloss:0.44056
[13]	validation_0-logloss:0.43334
[14]	validation_0-logloss:0.42246
[15]	validation_0-logloss:0.41532
[16]	validation_0-logloss:0.41076
[17]	validation_0-logloss:0.40651
[18]	validation_0-logloss:0.40552
[19]	validation_0-logloss:0.39891
[20]	validation_0-logloss:0.39862
[21]	validation_0-logloss:0.39557
[22]	validation_0-logloss:0.39250
[23]	validation_0-logloss:0.38825
[24]	validation_0-logloss:0.38992
[25]	validation_0-logloss:0.38758
[26]	validation_0-logloss:0.38804
[27]	validation_0-logloss:0.38979
[28]	validation_0-logloss:0.39207
[29]	validation_0-loglos